In [ ]:
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer, util

# Load the lightweight spaCy English model for Part-Of-Speech (POS) tagging
nlp = spacy.load("en_core_web_sm")

# Dummy dataset containing original and emotion-shifted sentences
data = {
    'original': [
        'The devastating storm destroyed the house.',
        'I am crying about the terrible test results.',
        'He walked slowly and miserably to the car.'
    ],
    'transformed': [
        'The beautiful storm blessed the house.',
        'I am cheering about the amazing test results.',
        'He walked quickly and happily to the car.'
    ]
}

df = pd.DataFrame(data)
df.head()

,original,transformed
0,The devastating storm destroyed the house.,The beautiful storm blessed the house.
1,I am crying about the terrible test results.,I am cheering about the amazing test results.
2,He walked slowly and miserably to the car.,He walked quickly and happily to the car.


In [2]:
def mask_emotion_words(text):
    # Process the text with spaCy
    doc = nlp(text)
    
    # Filter out Adjectives (ADJ) and Adverbs (ADV)
    # We keep Nouns, Verbs, Pronouns, Prepositions, etc.
    masked_tokens = [token.text for token in doc if token.pos_ not in ['ADJ', 'ADV']]
    
    # Rejoin the structural words back into a single string
    return " ".join(masked_tokens)

# Apply the mask to both columns
df['masked_original'] = df['original'].apply(mask_emotion_words)
df['masked_transformed'] = df['transformed'].apply(mask_emotion_words)

print("Text masking complete.")

Text masking complete.


In [8]:
# Initialize the Sentence Transformer
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode only the MASKED sentences
emb_orig_masked = model.encode(df['masked_original'].tolist(), convert_to_tensor=True)
emb_trans_masked = model.encode(df['masked_transformed'].tolist(), convert_to_tensor=True)

emb_orig = model.encode(df['original'].tolist(), convert_to_tensor=True)
emb_trans = model.encode(df['transformed'].tolist(), convert_to_tensor=True)

# Calculate cosine similarities
cosine_scores_masked = util.cos_sim(emb_orig_masked, emb_trans_masked)
cosine_scores = util.cos_sim(emb_orig, emb_trans)

# Extract the diagonal (row-by-row similarity)
df['content_preservation_score'] = [cosine_scores[i][i].item() for i in range(len(df))]
df['content_preservation_score_masked'] = [cosine_scores_masked[i][i].item() for i in range(len(df))]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Display the results. Notice how the scores are very high 
# because the underlying structural words perfectly match.
display(df[['original', 'transformed', 'masked_original', 'masked_transformed', 'content_preservation_score', 'content_preservation_score_masked']])

,original,transformed,masked_original,masked_transformed,content_preservation_score,content_preservation_score_masked
0,The devastating storm destroyed the house.,The beautiful storm blessed the house.,The storm destroyed the house .,The storm blessed the house .,0.663852,0.757146
1,I am crying about the terrible test results.,I am cheering about the amazing test results.,I am crying about the test results .,I am cheering about the test results .,0.727604,0.757132
2,He walked slowly and miserably to the car.,He walked quickly and happily to the car.,He walked and to the car .,He walked and to the car .,0.701062,1.000000
